# SASRec Time-Aware BPI2012 Colab Train 01

Colab notebook for the first time-aware SASRec experiment on BPI 2012.

Goals:
- reuse the existing `refine_v3_ml50_do025` baseline results instead of retraining them
- train only the new time-aware runs with time-delta bucket embeddings
- compare baseline vs `8-bucket` vs `9-bucket` under both `NDCG@10` and `NDCG@5` model-selection criteria


In [ ]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012'
BASELINE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
TIMEAWARE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012'
TIMEAWARE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('BASELINE_NDCG5_OUTPUT_DIR:', BASELINE_NDCG5_OUTPUT_DIR)
print('TIMEAWARE_NDCG10_OUTPUT_DIR:', TIMEAWARE_NDCG10_OUTPUT_DIR)
print('TIMEAWARE_NDCG5_OUTPUT_DIR:', TIMEAWARE_NDCG5_OUTPUT_DIR)


In [ ]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$BASELINE_NDCG5_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG10_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG5_OUTPUT_DIR"


In [ ]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction


In [ ]:
# If you need the latest code from GitHub, uncomment below.
# %cd /content/time-aware-behavior-prediction
# !git pull


In [ ]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


In [ ]:
!pip install -r requirements_colab.txt


In [ ]:
!ls "$DATA_DIR"


In [ ]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


## Experiment design

Fixed baseline setting:
- `refine_v3_ml50_do025`
- `hidden_units=50, num_blocks=2, num_heads=1, maxlen=50, lr=0.001, dropout=0.25`
- seeds: `42`, `2024`

Comparison targets:
- baseline (reuse existing completed runs)
- time-aware `8-bucket`
- time-aware `9-bucket`

Bucket designs:
- `8-bucket`: `padding`, `first`, `zero-gap`, `(0,1m)`, `[1m,10m)`, `[10m,1h)`, `[1h,1d)`, `[>=1d]`
- `9-bucket`: `padding`, `first`, `zero-gap`, `(0,1m)`, `[1m,10m)`, `[10m,1h)`, `[1h,1d)`, `[1d,7d)`, `[>=7d]`


## Check existing baseline runs

These baseline runs should already exist and must not be retrained.


In [ ]:
from pathlib import Path

baseline_ndcg10_runs = [
    'refine_v3_ml50_do025_seed42',
    'refine_v3_ml50_do025_seed2024',
]
baseline_ndcg5_runs = [
    'refine_v3_ml50_do025_seed42_ndcg5',
    'refine_v3_ml50_do025_seed2024_ndcg5',
]

for label, output_dir, run_names in [
    ('Baseline NDCG@10', Path(BASELINE_NDCG10_OUTPUT_DIR), baseline_ndcg10_runs),
    ('Baseline NDCG@5', Path(BASELINE_NDCG5_OUTPUT_DIR), baseline_ndcg5_runs),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


## Check planned time-aware runs

Only train runs that are still missing.


In [ ]:
planned_ndcg10 = [
    'timeaware01_refine_v3_ml50_do025_seed42_b8',
    'timeaware01_refine_v3_ml50_do025_seed2024_b8',
    'timeaware01_refine_v3_ml50_do025_seed42_b9',
    'timeaware01_refine_v3_ml50_do025_seed2024_b9',
]
planned_ndcg5 = [
    'timeaware01_refine_v3_ml50_do025_seed42_b8_ndcg5',
    'timeaware01_refine_v3_ml50_do025_seed2024_b8_ndcg5',
    'timeaware01_refine_v3_ml50_do025_seed42_b9_ndcg5',
    'timeaware01_refine_v3_ml50_do025_seed2024_b9_ndcg5',
]

for label, output_dir, run_names in [
    ('Time-aware NDCG@10', Path(TIMEAWARE_NDCG10_OUTPUT_DIR), planned_ndcg10),
    ('Time-aware NDCG@5', Path(TIMEAWARE_NDCG5_OUTPUT_DIR), planned_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


## Train time-aware runs for `NDCG@10`

Run these cells only if the corresponding run directory does not already exist.


### timeaware01_refine_v3_ml50_do025_seed42_b8


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware01_refine_v3_ml50_do025_seed42_b8 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware01_refine_v3_ml50_do025_seed2024_b8


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware01_refine_v3_ml50_do025_seed2024_b8 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware01_refine_v3_ml50_do025_seed42_b9


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware01_refine_v3_ml50_do025_seed42_b9 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware01_refine_v3_ml50_do025_seed2024_b9


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware01_refine_v3_ml50_do025_seed2024_b9 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


## Train time-aware runs for `NDCG@5`

Run these cells only if the corresponding run directory does not already exist.


### timeaware01_refine_v3_ml50_do025_seed42_b8_ndcg5


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware01_refine_v3_ml50_do025_seed42_b8_ndcg5 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware01_refine_v3_ml50_do025_seed2024_b8_ndcg5


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware01_refine_v3_ml50_do025_seed2024_b8_ndcg5 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware01_refine_v3_ml50_do025_seed42_b9_ndcg5


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware01_refine_v3_ml50_do025_seed42_b9_ndcg5 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


### timeaware01_refine_v3_ml50_do025_seed2024_b9_ndcg5


In [ ]:
!python src/train_sasrec.py \
  --run_name timeaware01_refine_v3_ml50_do025_seed2024_b9_ndcg5 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_bpi2012_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


## Rebuild result tables from run folders

This avoids schema issues in `experiment_index.csv` and lets us combine old baseline runs with new time-aware runs safely.


In [ ]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_embedding': config.get('use_time_embedding', False),
            'time_bucket_boundaries': ','.join(str(x) for x in config.get('time_bucket_boundaries', [])),
            'time_bucket_count': config.get('time_bucket_count'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


## NDCG@10 comparison summary


In [ ]:
ndcg10_baseline_runs = [
    'refine_v3_ml50_do025_seed42',
    'refine_v3_ml50_do025_seed2024',
]
ndcg10_timeaware_runs = [
    'timeaware01_refine_v3_ml50_do025_seed42_b8',
    'timeaware01_refine_v3_ml50_do025_seed2024_b8',
    'timeaware01_refine_v3_ml50_do025_seed42_b9',
    'timeaware01_refine_v3_ml50_do025_seed2024_b9',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG10_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(ndcg10_baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['bucket_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(ndcg10_timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['bucket_variant'] = timeaware_subset['run_name'].apply(lambda x: 'b8' if '_b8' in x else 'b9')

df_ndcg10 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg10 = df_ndcg10.sort_values(['bucket_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg10[[
    'run_name', 'seed', 'bucket_variant', 'use_time_embedding', 'time_bucket_boundaries',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]]


In [ ]:
summary_ndcg10 = df_ndcg10.groupby('bucket_variant')[[
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg10


Interpretation guide for NDCG@10:
- first compare `best_valid_full_ndcg@10` and `best_test_full_ndcg@10` across `baseline`, `b8`, and `b9`
- then check whether sampled metrics and MRR show a similar trend
- because baseline is reused from existing runs, only the time-aware runs are newly trained here


## NDCG@5 comparison summary


In [ ]:
ndcg5_baseline_runs = [
    'refine_v3_ml50_do025_seed42_ndcg5',
    'refine_v3_ml50_do025_seed2024_ndcg5',
]
ndcg5_timeaware_runs = [
    'timeaware01_refine_v3_ml50_do025_seed42_b8_ndcg5',
    'timeaware01_refine_v3_ml50_do025_seed2024_b8_ndcg5',
    'timeaware01_refine_v3_ml50_do025_seed42_b9_ndcg5',
    'timeaware01_refine_v3_ml50_do025_seed2024_b9_ndcg5',
]

baseline_df = rebuild_df(BASELINE_NDCG5_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG5_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(ndcg5_baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['bucket_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(ndcg5_timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['bucket_variant'] = timeaware_subset['run_name'].apply(lambda x: 'b8' if '_b8' in x else 'b9')

df_ndcg5 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg5 = df_ndcg5.sort_values(['bucket_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg5[[
    'run_name', 'seed', 'bucket_variant', 'use_time_embedding', 'time_bucket_boundaries',
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]]


In [ ]:
summary_ndcg5 = df_ndcg5.groupby('bucket_variant')[[
    'best_valid_full_ndcg@5', 'best_test_full_ndcg@5',
    'best_valid_full_ndcg@10', 'best_test_full_ndcg@10',
    'best_valid_full_mrr', 'best_test_full_mrr',
    'best_valid_sampled_ndcg@5', 'best_test_sampled_ndcg@5',
    'best_valid_sampled_ndcg@10', 'best_test_sampled_ndcg@10',
    'best_valid_sampled_mrr', 'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg5


Interpretation guide for NDCG@5:
- first compare `best_valid_full_ndcg@5` and `best_test_full_ndcg@5` across `baseline`, `b8`, and `b9`
- then check whether sampled metrics and MRR show a similar trend
- because baseline is reused from existing runs, only the time-aware runs are newly trained here
